# <center>**Chapter 16 - Using Dropout Regularization in PyTorch Models**<center>

Theo ví dụ Sonar: dùng cùng dữ liệu và `StratifiedKFold` 10-fold để so sánh mô hình cơ sở, Dropout ở lớp đầu vào (visible layer) và Dropout ở các lớp ẩn. Dropout chỉ hoạt động khi gọi `model.train()`; trong `model.eval()`, lớp này không làm thay đổi đầu vào.

In [ ]:
import csv
from pathlib import Path

import numpy as np
import torch
torch.set_num_threads(1)  # Dùng một luồng CPU để mô hình nhỏ chạy hiệu quả hơn.

from torch import nn
from torch.optim import SGD
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [ ]:
def find_data_file(filename):
    start = Path.cwd().resolve()
    patterns = (f"data/**/{filename}", f"*/data/**/{filename}", f"*/*/data/**/{filename}")
    for root in (start, *start.parents):
        for pattern in patterns:
            matches = list(root.glob(pattern))
            if matches:
                return matches[0]
    raise FileNotFoundError(f"Cannot find {filename} near the current directory")

with find_data_file("sonar.all-data").open(newline="", encoding="utf-8-sig") as file:
    rows = list(csv.reader(file))
X_raw = np.asarray([[float(value) for value in row[:60]] for row in rows], dtype=np.float32)
labels = LabelEncoder().fit_transform([row[60] for row in rows])
X = torch.tensor(X_raw, dtype=torch.float32)
y = torch.tensor(labels, dtype=torch.float32).reshape(-1, 1)

In [3]:
class SonarMLP(nn.Module):
    def __init__(self, dropout_position="none"):
        super().__init__()
        self.input_dropout = nn.Dropout(0.2) if dropout_position == "input" else nn.Identity()
        self.hidden_dropout1 = nn.Dropout(0.2) if dropout_position == "hidden" else nn.Identity()
        self.hidden_dropout2 = nn.Dropout(0.2) if dropout_position == "hidden" else nn.Identity()
        self.layer1 = nn.Linear(60, 60)
        self.layer2 = nn.Linear(60, 30)
        self.output = nn.Linear(30, 1)
        self.activation = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, features):
        values = self.input_dropout(features)
        values = self.hidden_dropout1(self.activation(self.layer1(values)))
        values = self.hidden_dropout2(self.activation(self.layer2(values)))
        return self.sigmoid(self.output(values))

In [4]:
def cross_validate(position, folds, epochs=300, batch_size=16):
    fold_accuracies = []
    for fold, (train_indices, validation_indices) in enumerate(folds, start=1):
        torch.manual_seed(100 + fold)
        scaler = StandardScaler().fit(X_raw[train_indices])
        X_train = torch.tensor(scaler.transform(X_raw[train_indices]), dtype=torch.float32)
        X_validation = torch.tensor(scaler.transform(X_raw[validation_indices]), dtype=torch.float32)
        y_train, y_validation = y[train_indices], y[validation_indices]
        model = SonarMLP(position)
        optimizer = SGD(model.parameters(), lr=0.01, momentum=0.8)
        loss_fn = nn.BCELoss()
        for _ in range(epochs):
            model.train()
            for start in range(0, len(X_train), batch_size):
                features = X_train[start:start + batch_size]
                targets = y_train[start:start + batch_size]
                loss = loss_fn(model(features), targets)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        model.eval()
        with torch.no_grad():
            accuracy = ((model(X_validation).round() == y_validation).float().mean().item())
        fold_accuracies.append(accuracy)
        print(f"{position:6s} | fold {fold:02d}/{len(folds)} | accuracy={accuracy:.2%}")
    return np.mean(fold_accuracies), np.std(fold_accuracies)

In [5]:
splitter = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
folds = list(splitter.split(X_raw, labels))
for position in ("none", "input", "hidden"):
    mean_accuracy, std_accuracy = cross_validate(position, folds)
    print(f"{position:6s} dropout | CV accuracy={mean_accuracy:.2%} (+/- {std_accuracy:.2%})\n")

none   | fold 01/10 | accuracy=100.00%
none   | fold 02/10 | accuracy=71.43%
none   | fold 03/10 | accuracy=85.71%
none   | fold 04/10 | accuracy=95.24%
none   | fold 05/10 | accuracy=80.95%
none   | fold 06/10 | accuracy=100.00%
none   | fold 07/10 | accuracy=90.48%
none   | fold 08/10 | accuracy=85.71%
none   | fold 09/10 | accuracy=80.00%
none   | fold 10/10 | accuracy=90.00%
none   dropout | CV accuracy=87.95% (+/- 8.66%)

input  | fold 01/10 | accuracy=90.48%
input  | fold 02/10 | accuracy=76.19%
input  | fold 03/10 | accuracy=90.48%
input  | fold 04/10 | accuracy=85.71%
input  | fold 05/10 | accuracy=85.71%
input  | fold 06/10 | accuracy=95.24%
input  | fold 07/10 | accuracy=90.48%
input  | fold 08/10 | accuracy=85.71%
input  | fold 09/10 | accuracy=70.00%
input  | fold 10/10 | accuracy=95.00%
input  dropout | CV accuracy=86.50% (+/- 7.59%)

hidden | fold 01/10 | accuracy=90.48%
hidden | fold 02/10 | accuracy=76.19%
hidden | fold 03/10 | accuracy=85.71%
hidden | fold 04/10 | accu

Kết quả 10-fold được tổng hợp thành trung bình và độ lệch chuẩn accuracy để so sánh ba cấu hình Dropout.

# <center>**HẾT**<center>